# epics

Generate db file:

```bash
$ ./harmonica-make-epics-db -c ../lattice/elettra.yaml -o elettra.db

```

Start epics server:

```bash
$ ./harmonica-start-epics-db elettra.db
```

Load data:

```bash
$ ./harmonica-load-epics-db -c ../lattice/elettra.yaml -tf ../data/reference.json.gz -n --sigma_x=0.000010 --sigma_y=0.000010
```

In [ ]:
# Test

from harmonica.cs import factory

cs = factory(target='epics')

print(cs.get("BPM:FREQUENCY:MODEL:X"))
print(cs.get("BPM:FREQUENCY:MODEL:Y"))

# tango

Generate db file:

```bash
$ ./harmonica-make-tango-ds -c ../lattice/elettra.yaml

```

Start tango server:

```bash
$ PORT=12345
$ LIST=$(paste -sd, device.list)
$ ./harmonica-start-tango-ds tbt -nodb -dlist "${LIST}" -ORBendPoint "giop:tcp:0.0.0.0:${PORT}"

```

Load data:

```bash
$ ./harmonica-load-tango-ds -c ../lattice/elettra.yaml -d device.yaml -t -f ../data/reference.json.gz -n --sigma_x=0.00001 --sigma_y=0.00001
```

In [ ]:
# Test

from harmonica.cs import factory

cs = factory(target='tango')

print(cs.get("BPM:FREQUENCY:MODEL:X"))
print(cs.get("BPM:FREQUENCY:MODEL:Y"))

In [ ]:
# Run without translation of epics to tango names

print(cs.get("bpm/harmonica/global/frequency_model_x", epics=False))
print(cs.get("bpm/harmonica/global/frequency_model_y", epics=False))

# tbt

Loads turn-by-turn (TbT) BPM waveforms for a selected plane, applies optional preprocessing/filtering (mean/median/normalize, SVD/Hankel), then plots and/or saves the processed TbT matrix.

In [ ]:
%run -i ./tbt.py -h

In [ ]:
%run -i ./tbt.py --plane=x --length=1024 --mean --only '^BPM_S01_.*$' --plot --tango

In [ ]:
%run -i ./tbt.py --plane=y --length=1024 --mean --only '^BPM_S01_.*$' --plot --tango

# trajectory

Builds a trajectory view from TbT data by stacking turns across BPMs, supports trajectory-difference plots (including BPM pair differences), and can compare against a saved reference trajectory.

In [ ]:
%run -i ./trajectory.py -h

In [ ]:
%run -i ./trajectory.py --plane=x --length=4 --plot --tango

# orbit

Computes the closed-orbit-like profile from TbT data (mean or median per BPM over turns), then plots/saves BPM orbit vs position.

In [ ]:
%run -i ./orbit.py -h

In [ ]:
%run -i ./orbit.py --plane=x --length=256 --plot --tango

# noise

Estimates per-BPM noise (rank/sigma) from TbT data, optionally via plain standard deviation, with optional autocorrelation visualization and PV update of noise values.

In [ ]:
%run -i ./noise.py -h

In [ ]:
%run -i ./noise.py --plane=x --length=512 --mean --plot --update --tango

In [ ]:
%run -i ./noise.py --plane=y --length=512 --mean --plot --update --tango

In [ ]:
# note, high rank might indicate potential power supply ripple

# spectrum

Computes per-BPM spectral content (FFT or FFRFT region), with optional windowing/filtering/log scaling/flip, heatmap and average-spectrum plotting, and peak extraction.

In [ ]:
%run -i ./spectrum.py -h

In [ ]:
%run -i ./spectrum.py --plane=x --length=1024 --mean --filter=svd --window=1.0 --pad=1024 --f_min=0.2 --f_max=0.4  --plot --map --log --average --peaks 1 --tango

In [ ]:
# demo (noise)

import numpy
from harmonica.cs import factory

In [ ]:
# Load original signal

signal = cs.get('BPM:BPM_S02_08:DATA:X')

In [ ]:
# Set signal with large leverl of noise
# Re-run the noise scripts to observer outlier BPM

cs.set('BPM:BPM_S02_08:DATA:X', signal + 0.01*numpy.random.randn(*signal.shape))

In [ ]:
# Restore orignal signal

cs.set('BPM:BPM_S02_08:DATA:X', signal)

# spectrum (all)

Computes a combined spectrum across selected BPMs (including optional NUFFT timing modes), useful for collective tune-feature extraction instead of per-BPM spectr.

In [ ]:
%run -i ./spectrum_all.py -h

In [ ]:
%run -i ./spectrum_all.py --plane=x --length=16 --normalize --window=0.0 --f_min=30.0 --f_max=35.0 --nufft --time=phase --plot --tango

In [ ]:
%run -i ./spectrum_all.py --plane=y --length=16 --normalize --window=0.0 --f_min=6.0 --f_max=12.0 --nufft --time=position --plot --tango

# frequency

Estimates per-BPM betatron frequency using selected estimators (FFT/FFRFT/parabola), performs optional cleaning/weighting, then reports/plots BPM frequencies plus global center/spread.

In [ ]:
%run -i ./frequency.py -h

In [ ]:
%run -i ./frequency.py --plane=x --length=256 --normalize --filter=svd --window=1.0 --method=parabola --clean --process=noise --plot --update --tango

In [ ]:
%run -i ./frequency.py --plane=y --length=256 --normalize --filter=svd --window=1.0 --method=parabola --clean --process=noise --plot --update --tango

# frequency (shift)

Repeats frequency estimation on shifted windows of the same TbT data to assess stability and uncertainty, then aggregates per-BPM and global statistics.

In [ ]:
%run -i ./frequency_shift.py -h

In [ ]:
%run -i ./frequency_shift.py --plane=x --length=128 --load=256 --shift=4 --mean --filter=svd --window=0.0 --pad=128 --method=parabola --process=noise --clean --plot --update --tango

In [ ]:
%run -i ./frequency_shift.py --plane=y --length=128 --load=256 --shift=4 --mean --filter=svd --window=0.0 --pad=128 --method=parabola --process=noise --clean --plot --update --tango

# frequency (all)

Computes mixed/joined frequency over shifted samples and selected BPMs, producing global frequency statistics and optional plots.

In [ ]:
%run -i ./frequency_all.py -h

In [ ]:
%run -i ./frequency_all.py --plane=x --length=16 --load=32 --window=1.0 --shift=1 --f_min=30.0 --f_max=35.0 --nufft --time=phase --plot --tango

# amplitude

Estimates harmonic amplitude per BPM (with optional propagated errors, shifted-sample mode, or DHT), supports coupled-plane mode, and can compute SNR from stored noise.

In [ ]:
%run -i ./amplitude.py -h

In [ ]:
%run -i ./amplitude.py --plane=x --length=32 --load=64 --mean --filter=svd --window=1.0 --shift --delta=4 --method=noise --snr --lg --plot --update --tango

In [ ]:
%run -i ./amplitude.py --plane=y --length=32 --load=64 --mean  --filter=svd --window=1.0 --shift --delta=4 --method=noise --snr --lg --plot --update --tango

# phase

Estimates harmonic phase per BPM (or DHT-based), with optional monotonic phase unfolding and adjacent phase-advance analysis against model/reference.

In [ ]:
%run -i ./phase.py -h

In [ ]:
%run -i ./phase.py --plane=x --length=256 --load=512 --normalize --filter=svd --window=1.0 --shift --delta=8 --method=noise --plot --advance --monotonic  --update --tango

In [ ]:
%run -i ./phase.py --plane=y --length=256 --load=512 --normalize --filter=svd --window=1.0 --shift --delta=8 --method=noise --plot --advance --monotonic  --update --tango

# check (synchronization)

Performs phase-synchronization consistency checks by comparing measured phase behavior against model expectations and highlighting problematic BPMs.

In [ ]:
%run -i ./check.py -h

In [ ]:
%run -i ./check.py -r --plane=x --length=256 --mean --window=0.0 --plot --tango

In [ ]:
%run -i ./check.py -r --plane=y --length=256 --mean --window=0.0 --plot --tango

In [ ]:
# demo (synchronization)

from harmonica.cs import factory
cs = factory(target='tango')

In [ ]:
# Change BPM first turn and re-run script

cs.set('BPM:BPM_S01_07:RISE', 1)

In [ ]:
# Restore

cs.set('BPM:BPM_S01_07:RISE', 0)

# twiss_amplitude

Reconstructs Twiss beta functions from amplitude-based data, compares against model beta, and visualizes action/beta agreement and residuals.

In [ ]:
%run -i ./twiss_amplitude.py -h

In [ ]:
%run -i ./twiss_amplitude.py --model='../lattice/elettra.yaml' --clean --plot --update --tango

# twiss_phase

Reconstructs Twiss from phase advance (with optional amplitude overlays), including model-vs-measured comparisons, virtual BPM handling, and residual/error views.

In [ ]:
%run -i ./twiss_phase.py -h

In [ ]:
%run -i ./twiss_phase.py --model='../lattice/elettra.yaml' --limit=5 --clean --plot --amplitude --update --tango

# action

Uses model + measured frequency/amplitude/phase data to compute transverse actions (`Jx`, `Jy`), including cleaned/weighted action summaries and outlier visualization.

In [ ]:
%run -i ./action.py -h

In [ ]:
%run -i ./action.py --model='../lattice/elettra.yaml' --clean --plot --update --tango

# ratio

Computes and plots the ratios `BX_A/BX` and `BY_A/BY` (amplitude-derived vs phase-derived Twiss beta), including propagated ratio uncertainties and current-based quality context.

In [ ]:
%run -i ./ratio.py -h

In [ ]:
%run -i ./ratio.py --plot --update --tango

In [ ]:
# demo (calibration)

from harmonica.cs import factory
cs = factory(target='tango')

In [ ]:
# Get original signal

signal = cs.get('BPM:BPM_S02_08:DATA:X')

In [ ]:
# Set signal with calibraton error
# Re-run scripts for amplitude, phase, twiss parameters and ratio

cs.set('BPM:BPM_S02_08:DATA:X', 1.5*signal)

In [ ]:
# Restore

cs.set('BPM:BPM_S02_08:DATA:X', 1.0*signal)